In [ ]:
import jupyter_black
jupyter_black.load()

import gzip
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from ccl_science_data.common import PUBY, iter_snap_items, read_full_df, snap_dir, EntC, DN

In [ ]:
snap_dir = Path("/mnt/alpha-solid/oa-snapshot-2025-02")

In [ ]:
inst_names = [
    "Massachusetts Institute of Technology",
    "Corvinus University of Budapest",
    "Utrecht University",
    "Zhejiang University",
    "Toulouse School of Economics",
    "Northeastern University",
    "University of Cambridge",
    "University of Chile",
    "Hungarian Academy of Sciences",
    "Bocconi University",
    "Stanford University",
    "California Institute of Technology",
    "Howard Hughes Medical Institute",
]

In [ ]:
insts = []
for _, isnap in iter_snap_items(EntC.INSTITUTIONS, snap_dir):
    if any(e.encode() in isnap for e in inst_names):
        insts.append(json.loads(isnap))

In [ ]:
intro_oa_ids = (
    pd.DataFrame([{k: i[k] for k in ["id", DN]} for i in insts])
    .drop_duplicates(subset=DN)
    .set_index(DN)
    .loc[inst_names, "id"]
    .tolist()
)

In [ ]:
len(inst_names)

In [ ]:
micro_oa_ids = intro_oa_ids[:8]

In [ ]:
nano_oa_ids = intro_oa_ids[:3]

In [ ]:
test_root = Path(os.environ["OA_TEST_ROOT"])

In [ ]:
test_root.mkdir(exist_ok=True)

In [ ]:
mini_snap = test_root / "mini-snapshot"
micro_snap = test_root / "micro-snapshot"
nano_snap = test_root / "nano-snapshot"

In [ ]:
rest_ents = [
    EntC.CONCEPTS,
    EntC.DOMAINS,
    EntC.FIELDS,
    EntC.SUBFIELDS,
    EntC.PUBLISHERS,
    EntC.TOPICS,
]

In [ ]:
def entity_filter(e: str, src_dir, target_dir, filter_fun=lambda x: True):
    rdir = src_dir / "data" / e
    jsfiles = []
    for subd in rdir.iterdir():
        if not subd.is_dir():
            continue
        jsfiles.extend(subd.iterdir())
    for jsf in tqdm(jsfiles, e):
        olines = []
        with gzip.open(jsf) as gzp:
            for gl in gzp:
                jso = json.loads(gl)
                # return jso
                if filter_fun(jso):
                    # return jso
                    olines.append(gl)
        if len(olines) > 0:
            out_p = target_dir / jsf.relative_to(src_dir)
            out_p.parent.mkdir(exist_ok=True, parents=True)
            out_p.write_bytes(gzip.compress(b"".join(olines)))

In [ ]:
class WFler:
    def __init__(self, oa_ids, minc=0, miny=0):
        self.oa_ids = oa_ids
        self.minc = minc
        self.miny = miny
        self.insts = []
        self.authors = []
        self.sources = []

    def __call__(self, jso):
        if jso.get("cited_by_count", 0) < self.minc:
            return False
        if jso[PUBY] < self.miny:
            return False
        for a in jso.get("authorships", []):
            if any(i["id"] in self.oa_ids for i in a["institutions"]):
                self.insts.extend(i["id"] for i in a["institutions"])
                self.sources.extend(
                    (l["source"] or {}).get("id") for l in jso["locations"]
                )
                self.authors.append(a["author"]["id"])
                return True
        return False

    def snowball(self, src_snap, target_snap):
        _sets = list(map(set, [self.authors, self.insts, self.sources]))
        for vset, e in zip(_sets, [EntC.AUTHORS, EntC.INSTITUTIONS, EntC.SOURCES]):
            print(e, len(vset) / 1e6)
            entity_filter(e, src_snap, target_snap, lambda e: e["id"] in vset)

        for e in rest_ents:
            entity_filter(e, src_snap, target_snap)

In [ ]:
wfler = WFler(intro_oa_ids)
entity_filter(EntC.WORKS, snap_dir, mini_snap, wfler)

In [ ]:
wfler.snowball(snap_dir, mini_snap)

In [ ]:
micro_wfler = WFler(micro_oa_ids, 10, 2010)
entity_filter(EntC.WORKS, mini_snap, micro_snap, micro_wfler)

In [ ]:
micro_wfler.snowball(mini_snap, micro_snap)

In [ ]:
nano_wfler = WFler(nano_oa_ids, 15, 2016)
entity_filter(EntC.WORKS, micro_snap, nano_snap, nano_wfler)

In [ ]:
nano_wfler.snowball(micro_snap, nano_snap)